# Visium HD subsampling — family table, nested replicate subsampling, QC

Pipeline (raw `molecule_info.h5` → per-depth `.h5ad` runs + QC + handoff schema):
1. **Cell 1** — config (paths, knobs) and load the deep reference AnnData.
2. **Cell 2** — stream `molecule_info.h5`, map 2µm squares → segmented cells via the barcode-mapping parquet, and write a per-UMI gene-aware "family" table to parquet shards.
3. **Cell 3** — nested, replicate-aware read-level subsampling across an RPC (reads-per-cell) grid; writes one `.h5ad` per (target_rpc, replicate) with per-cell reads/UMIs/saturation.
4. **Cell 4** — QC metrics (per-replicate + aggregated mean±SD), QC plots, and the `artifact_schema.json` consumed by the downstream stability-metrics notebook (`analysis_sub.ipynb`).

Edit the `CONFIG` block at the top of Cell 1 to point at your sample before running. This notebook stops at subsampling + QC; clustering/DE stability metrics live in the companion `analysis_sub.ipynb` notebook, which reads the `artifact_schema.json` written by Cell 4.


In [ ]:
# ============================================
# Cell 1: Config, parameters, and reference load
# ============================================
import os, time, math, sys, json, re, gc, warnings, hashlib
from typing import Dict, Tuple
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
from scipy import sparse
import h5py
import scanpy as sc

warnings.filterwarnings("ignore", category=UserWarning)
pl.Config.set_fmt_str_lengths(120)

# -------------------- User paths (edit per sample) --------------------
analysis_dir = "/path/to/your/project"          # root workspace for this sample's subsampling outputs
sample_name  = "SAMPLE"
REF_ADATA    = "/path/to/your/reference.h5ad"   # deep/full reference AnnData (segmented cells)

# Space Ranger v4+ output (single slide, deep sequencing run)
OUTS_DIR        = "/path/to/your/spaceranger/outs"
MOLECULE_INFO   = os.path.join(OUTS_DIR, "molecule_info.h5")          # required
MAPPING_PARQUET = os.path.join(OUTS_DIR, "barcode_mappings.parquet")  # required for 2um barcode -> segmented cell

# Workspace for intermediate shards & outputs
OUT_DIR = os.path.join(analysis_dir, sample_name)
os.makedirs(OUT_DIR, exist_ok=True)
SAVE_DIR = os.path.join(OUT_DIR, "runs_rpc_clamped")
os.makedirs(SAVE_DIR, exist_ok=True)

# -------------------- General parameters --------------------
# molecule_info streaming / shard writing
CHUNK_MOLECULES = 20_000_000                  # tune based on RAM; >=5M usually safe
KEEP_FEATURE_TYPE_SUBSTR = "gene expression"  # filter features/rows to this type (case-insensitive)

# Mapping parquet columns (Space Ranger v4+ VisiumHD cell segmentation)
CB_COL = "square_002um"     # 2um barcode / square id
CELLID_COL = "cell_id"      # segmented cell id
IN_CELL_COL = "in_cell"     # boolean; 2um square lies inside segmented cell
STRIP_CB_SUFFIX = True      # strip "-1" from barcodes when matching

# Output shard manifest
SHARD_GLOB = os.path.join(OUT_DIR, "family_geneaware_shard_*.parquet")
MANIFEST = os.path.join(OUT_DIR, "family_geneaware_shards.txt")

# -------------------- Sanity checks & reference load --------------------
assert os.path.exists(MOLECULE_INFO), f"Missing file: {MOLECULE_INFO}"
assert os.path.exists(MAPPING_PARQUET), f"Missing file: {MAPPING_PARQUET}"
assert os.path.exists(REF_ADATA), f"Missing file: {REF_ADATA}"

# Load the deep reference AnnData (segmented cells)
adata = sc.read_h5ad(REF_ADATA)
assert adata.n_obs > 0 and adata.n_vars > 0, "Empty AnnData loaded."


In [ ]:

# ============================================
# Cell 2: Build per-UMI, gene-aware family table from molecule_info.h5
# ============================================

def _strip_dash1(s: str) -> str:
    s = "" if s is None else str(s)
    return s[:-2] if s.endswith("-1") else s

def _strip_version(x: str) -> str:
    """Remove Ensembl version suffix, e.g., ENSG000001.13 -> ENSG000001."""
    x = "" if x is None else str(x)
    return x.split(".")[0] if "." in x else x

def build_gene_indexers(adata) -> Tuple[Dict[str, int], Dict[str, int]]:
    """
    Return two dicts:
      - id_to_idx: versionless stable ID (Ensembl-like) -> var index
      - sym_to_idx: gene symbol/name -> var index
    Priority will be ID first, then symbol as fallback.
    """
    var = adata.var.copy()
    # Try to detect common gene id / symbol columns
    candidate_id_cols = [c for c in ["gene_ids", "gene_id", "feature_id", "ensembl_id"] if c in var.columns]
    candidate_sym_cols = [c for c in ["gene_symbols", "gene_symbol", "gene_name", "symbol"] if c in var.columns]

    # Build ID mapper (versionless)
    id_to_idx = {}
    if candidate_id_cols:
        id_col = candidate_id_cols[0]
        ids = var[id_col].astype(str).map(_strip_version)
        id_to_idx = {ids.iat[i]: i for i in range(len(ids))}
    else:
        # If var_names are IDs (common in Ensembl-based matrices)
        vn = pd.Index(adata.var_names.astype(str).map(_strip_version))
        # Only keep entries that look like Ensembl (start with ENS)
        id_to_idx = {vn[i]: i for i in range(len(vn)) if vn[i].startswith("ENS")}

    # Build symbol mapper (fallback)
    sym_to_idx = {}
    if candidate_sym_cols:
        sym_col = candidate_sym_cols[0]
        syms = var[sym_col].astype(str)
        sym_to_idx = {syms.iat[i]: i for i in range(len(syms))}
    else:
        # If var_names are symbols (typical when IDs are not present)
        vn = pd.Index(adata.var_names.astype(str))
        sym_to_idx = {vn[i]: i for i in range(len(vn))}

    return id_to_idx, sym_to_idx

# ----------------------------
# 1) Build 2um barcode -> segmented cell_idx map (only in-cell + non-empty cell_id)
# ----------------------------
t0 = time.time()
print("[map] loading barcode_mappings.parquet ...")
lf = pl.scan_parquet(MAPPING_PARQUET)
cols = set(lf.collect_schema().names())
for c in [CB_COL, CELLID_COL, IN_CELL_COL]:
    assert c in cols, f"Column '{c}' missing in mapping parquet."

lf2 = (
    lf.filter(pl.col(IN_CELL_COL) == True)
      .filter(pl.col(CELLID_COL).is_not_null() & (pl.col(CELLID_COL) != ""))
)

if STRIP_CB_SUFFIX:
    lf2 = lf2.with_columns(
        pl.col(CB_COL).cast(pl.Utf8).map_elements(_strip_dash1, return_dtype=pl.Utf8).alias("CB_norm")
    )
    CB_KEY = "CB_norm"
else:
    lf2 = lf2.with_columns(pl.col(CB_COL).cast(pl.Utf8).alias("CB_norm"))
    CB_KEY = CB_COL

# Map cell_id -> adata index
cellid_to_idx = {str(cid): i for i, cid in enumerate(adata.obs_names.astype(str))}
mp = lf2.select([CB_KEY, CELLID_COL]).collect().to_pandas()
mp[CELLID_COL] = mp[CELLID_COL].astype(str)
mp = mp[mp[CELLID_COL].isin(cellid_to_idx.keys())]

cb_to_cellidx: Dict[str, int] = {}
for cb, cid in zip(mp[CB_KEY].tolist(), mp[CELLID_COL].tolist()):
    if cb not in cb_to_cellidx:
        cb_to_cellidx[cb] = cellid_to_idx[cid]

print(f"[map] mapped CB→cell: {len(cb_to_cellidx):,} unique 2um barcodes in {time.time()-t0:.1f}s")

# ----------------------------
# 2) Feature (gene) mapping: molecule_info features -> adata.var indices
# ----------------------------
print("[feat] loading features from molecule_info.h5 ...")
with h5py.File(MOLECULE_INFO, "r") as f:
    feat_id = f["features/id"][:]            # bytes (often Ensembl)
    feat_nm = f["features/name"][:]          # bytes (gene symbol)
    feat_tp = f["features/feature_type"][:]  # bytes

feat_id = np.array([x.decode("utf-8", "ignore") for x in feat_id], dtype=object)
feat_nm = np.array([x.decode("utf-8", "ignore") for x in feat_nm], dtype=object)
feat_tp = np.array([x.decode("utf-8", "ignore") for x in feat_tp], dtype=object)

u, c = np.unique(feat_tp, return_counts=True)
print("[feat] feature_type counts:")
for t, n in zip(u, c):
    print(f"       {t}: {n:,}")

keep_mask_type = np.array([KEEP_FEATURE_TYPE_SUBSTR.lower() in str(t).lower() for t in feat_tp], dtype=bool)
print(f"[feat] keeping types with '{KEEP_FEATURE_TYPE_SUBSTR}': {keep_mask_type.sum():,}/{feat_tp.size:,}")

# Build gene indexers from the reference AnnData
id_to_idx, sym_to_idx = build_gene_indexers(adata)

# Map features to var indices (prefer ID first, then symbol)
feat_to_gene_idx = np.full(feat_id.size, -1, dtype=np.int32)
hits_id = hits_sym = 0
for j in range(feat_id.size):
    if not keep_mask_type[j]:
        continue
    gi = -1
    fid = _strip_version(feat_id[j])
    if fid and fid in id_to_idx:
        gi = id_to_idx[fid]; hits_id += 1
    else:
        gnm = feat_nm[j]
        if gnm and gnm in sym_to_idx:
            gi = sym_to_idx[gnm]; hits_sym += 1
    feat_to_gene_idx[j] = gi

n_unmapped = int((feat_to_gene_idx < 0).sum())
print(f"[feat] mapped features → adata.var | ID={hits_id:,}, Symbol={hits_sym:,}, Unmapped={n_unmapped:,}")

# 3) Barcode index (molecule_info) -> adata cell_idx
print("[bar] loading molecule_info barcodes ...")
with h5py.File(MOLECULE_INFO, "r") as f:
    barcodes = f["barcodes"][:]  # bytes
barcodes = [x.decode("utf-8", "ignore") for x in barcodes]
if STRIP_CB_SUFFIX:
    barcodes = [_strip_dash1(s) for s in barcodes]

n_barcodes = len(barcodes)
barcode_to_cell_idx = np.full(n_barcodes, -1, dtype=np.int32)
hit = 0
for i, s in enumerate(barcodes):
    ci = cb_to_cellidx.get(s, -1)
    barcode_to_cell_idx[i] = ci
    if ci >= 0:
        hit += 1
print(f"[bar] barcodes mapped to a segmented cell: {hit:,} / {n_barcodes:,}")

# 4) Stream molecule_info rows -> per-UMI family shards (cell_idx, gene_idx, umi_reads, umi_id)
print("[fam] streaming molecules to per-UMI shards ...")
shards = []
with h5py.File(MOLECULE_INFO, "r") as f:
    n_mol = f["barcode_idx"].shape[0]
    n_chunks = math.ceil(n_mol / CHUNK_MOLECULES)
    print(f"[fam] molecules: {n_mol:,} | chunks: {n_chunks}")

    for k in range(n_chunks):
        start = k * CHUNK_MOLECULES
        end   = min((k + 1) * CHUNK_MOLECULES, n_mol)
        t1 = time.time()

        # Load chunk
        bc_idx  = f["barcode_idx"][start:end].astype(np.int64, copy=False)
        ft_idx  = f["feature_idx"][start:end].astype(np.int64, copy=False)
        umi     = f["umi"][start:end].astype(np.uint32, copy=False)
        reads   = f["count"][start:end].astype(np.int32, copy=False)

        # Map indices
        cells = barcode_to_cell_idx[bc_idx]
        genes = feat_to_gene_idx[ft_idx]

        # Keep only: in segmented cell, mapped gene, positive reads
        mask = (cells >= 0) & (genes >= 0) & (reads > 0)
        kept = int(mask.sum())

        if kept > 0:
            df_pl = pl.DataFrame({
                "cell_idx":  pl.Series(cells[mask].astype(np.int32)),
                "gene_idx":  pl.Series(genes[mask].astype(np.int32)),
                "umi_reads": pl.Series(reads[mask].astype(np.int32)),   # per-UMI-family read count (confident gene-mapped)
                "umi_id":    pl.Series(umi[mask].astype(np.uint32)),    # optional, useful for QA
            })
            shard = os.path.join(OUT_DIR, f"family_geneaware_shard_{k:04d}.parquet")
            df_pl.write_parquet(shard, compression="zstd")
            shards.append(shard)

        rate = (end - start) / max(1e-6, (time.time() - t1))
        print(f"[fam] chunk {k+1}/{n_chunks} rows {start:,}..{end-1:,} | kept={kept:,} | rate≈{rate:,.0f}/s")

if not shards:
    raise RuntimeError("No per-UMI families were collected; check mapping/filters.")

# Write a manifest so downstream can glob safely (and you have a record)
with open(MANIFEST, "w") as fh:
    for s in shards:
        fh.write(s + "\n")

# Quick QC via lazy scan (no full load)
lf_all = pl.scan_parquet(SHARD_GLOB)
qc = (
    lf_all.select([
        pl.len().alias("n_families"),
        pl.col("umi_reads").sum().alias("total_reads"),
        pl.col("cell_idx").n_unique().alias("n_cells_nonempty"),
        pl.col("gene_idx").n_unique().alias("n_genes_nonempty"),
        pl.col("umi_reads").mean().alias("mean_reads_per_family"),
    ])
    .collect()
    .to_pandas()
    .iloc[0]
    .to_dict()
)

fmt_qc = {k: (int(v) if isinstance(v, (np.integer, int)) else (round(float(v), 3))) for k, v in qc.items()}
print("[fam][QC]", fmt_qc)
print("[done] shards written. Manifest:", MANIFEST)


In [ ]:
# ============================================
# Cell 3: Global, nested read-level subsampling with replicates
#         RPC grid (feasible max), N_REPS per target
#         Outputs per-depth .h5ad with per-cell reads/UMIs/saturation
# ============================================
LAYER_LOGNORM = "lognorm"

# Grid settings (depth levels; inclusive of max)
N_STEPS = 10        # equally spaced RPC targets (incl. max)
N_REPS  = 3         # number of replicate subsamples PER target RPC

# RNG seed (replicate-level; nested across depths within a replicate)
RNG_SEED_BASE = 12345

# Micro-batching of shard rows (families) to limit peak RAM when vectorizing
FAMILY_BATCH_SIZE = 2_000_000

# ----------------------------------------

def _ensure_lognorm(ad, layer=LAYER_LOGNORM):
    if layer not in ad.layers:
        norm = sc.pp.normalize_total(ad, target_sum=1e4, inplace=False)
        ad.layers[layer] = norm["X"]
        sc.pp.log1p(ad, layer=layer)

def _read_shard(path):
    """Expect columns: cell_idx, gene_idx, umi_reads  (or legacy family_reads)."""
    try:
        df = pl.read_parquet(path, columns=["cell_idx","gene_idx","umi_reads"]).to_pandas()
    except Exception:
        df = pl.read_parquet(path, columns=["cell_idx","gene_idx","family_reads"]).to_pandas()
        df.rename(columns={"family_reads":"umi_reads"}, inplace=True)
    # dtypes
    df["cell_idx"]  = df["cell_idx"].astype(np.int32, copy=False)
    df["gene_idx"]  = df["gene_idx"].astype(np.int32, copy=False)
    df["umi_reads"] = df["umi_reads"].astype(np.int32, copy=False)
    return df

def _build_counts_matrix(n_cells, n_genes, grouped_triplets):
    if not grouped_triplets:
        return sparse.csr_matrix((n_cells, n_genes), dtype=np.int32)
    if len(grouped_triplets) == 1:
        rows, cols, dat = grouped_triplets[0]
    else:
        rows = np.concatenate([t[0] for t in grouped_triplets])
        cols = np.concatenate([t[1] for t in grouped_triplets])
        dat  = np.concatenate([t[2] for t in grouped_triplets])
    return sparse.coo_matrix((dat, (rows, cols)), shape=(n_cells, n_genes), dtype=np.int32).tocsr()

def _load_manifest(p):
    with open(p, "r") as f:
        paths = [ln.strip() for ln in f if ln.strip()]
    if not paths:
        raise RuntimeError(f"Empty manifest: {p}")
    return paths

def _rng_for_rep(rep: int) -> np.random.Generator:
    # stable per-replicate RNG; ensures nestedness across depths within a rep
    h = int(hashlib.blake2b(f"rep|{rep}".encode(), digest_size=8).hexdigest(), 16) & 0x7FFFFFFF
    return np.random.default_rng(RNG_SEED_BASE + h)

# ---------- Load reference + shards ----------
ad_ref = sc.read_h5ad(REF_ADATA)
n_cells, n_genes = ad_ref.n_obs, ad_ref.n_vars
print(f"[ref] {n_cells:,} cells × {n_genes:,} genes")

shards = _load_manifest(MANIFEST)
print(f"[manifest] {len(shards)} shards")

# ---------- Compute total reads present in shards (we thin these reads) ----------
reads_in_shards = 0
families_in_shards = 0
t0 = time.time()
for i, sh in enumerate(shards, 1):
    try:
        s = pl.read_parquet(sh, columns=["umi_reads"]).select(pl.col("umi_reads").sum()).item()
    except Exception:
        s = pl.read_parquet(sh, columns=["family_reads"]).select(pl.col("family_reads").sum()).item()
    reads_in_shards += int(s)

    fam_ct = pl.read_parquet(sh, columns=["cell_idx"]).height
    families_in_shards += int(fam_ct)
    if i == 1 or i % 2 == 0 or i == len(shards):
        print(f"  [scan] shard {i:02d}/{len(shards)} | +reads={int(s):,} | cum_reads={reads_in_shards:,}")
print(f"[totals] reads_in_shards={reads_in_shards:,} | families={families_in_shards:,} | {time.time()-t0:.1f}s")

# ---------- Determine feasible max RPC and build equally spaced target list ----------
max_rpc = int(np.floor(reads_in_shards / n_cells))
print(f"[feasible] max_rpc={max_rpc:,} (reads_in_shards {reads_in_shards:,} / cells {n_cells:,})")

if max_rpc <= 0:
    target_rpcs = []
else:
    # Equally spaced integers from 50 to max_rpc (inclusive), with N_STEPS points.
    target_rpcs = sorted(np.unique(np.linspace(50, max_rpc, N_STEPS, dtype=int)).tolist())
    target_rpcs = [t for t in target_rpcs if t > 0]

print("[targets]", target_rpcs)

# Precompute p_keep (in ascending order) and maintain a mapping to target_rpc
p_for_target = []
for t in target_rpcs:
    p = float(t) * float(n_cells) / float(reads_in_shards)
    p_for_target.append(float(np.clip(p, 0.0, 1.0)))

# Build unique, ascending p-grid for nested sampling
p_grid = np.array(sorted(set(p_for_target)))
K = len(p_grid)

# Helper: map a p_target to index in p_grid (robust to float jitter / duplicates)
def _p_to_index(p_target: float) -> int:
    if K == 0:
        return 0
    j = int(np.argmin(np.abs(p_grid - p_target)))
    return j

# ---------- Subsampling with replicates (nested within each replicate across p_grid) ----------
index_rows = []

for rep in range(N_REPS):
    rng = _rng_for_rep(rep)
    print(f"\n[rep] {rep} | nested sampling over {K} depths (p_grid asc)")

    # Per-depth accumulators
    grouped_per_depth = [[] for _ in range(K)]                         # lists of (rows, cols, data)
    reads_kept_per_cell_list = [np.zeros(n_cells, dtype=np.int64) for _ in range(K)]
    umis_surv_per_cell_list  = [np.zeros(n_cells, dtype=np.int64) for _ in range(K)]
    kept_reads_total_list    = [0 for _ in range(K)]
    kept_umis_total_list     = [0 for _ in range(K)]

    # Stream all shards; update ALL depths in a nested manner
    for si, sh in enumerate(shards, 1):
        s0 = time.time()
        df  = _read_shard(sh)
        r   = df["umi_reads"].to_numpy()
        cel = df["cell_idx"].to_numpy()
        gen = df["gene_idx"].to_numpy()
        F = r.size

        if F == 0:
            print(f"  [shard {si:02d}/{len(shards)}] empty shard? skipping.")
            continue

        # Process in family micro-batches to bound memory
        for start in range(0, F, FAMILY_BATCH_SIZE):
            end = min(start + FAMILY_BATCH_SIZE, F)
            rb  = r[start:end]
            cb  = cel[start:end]
            gb  = gen[start:end]

            # Nested binomial ladder across p_grid:
            # k_j ~ Binom(r, p_j) with k_{j+1} = k_j + Binom(r - k_j, (p_{j+1}-p_j)/(1-p_j))
            k = None
            p_prev = 0.0

            for j, p in enumerate(p_grid):
                if j == 0:
                    k = rng.binomial(rb, p).astype(np.int32, copy=False)
                else:
                    remain = rb - k
                    denom = max(1e-12, 1.0 - p_prev)
                    q = (p - p_prev) / denom if p > p_prev else 0.0
                    q = float(np.clip(q, 0.0, 1.0))  # guard
                    if remain.dtype != np.int32:
                        remain = remain.astype(np.int32, copy=False)
                    k = k + rng.binomial(remain, q).astype(np.int32, copy=False)

                # per-cell kept reads
                if k.size:
                    pc_reads = (
                        pd.DataFrame({"cell_idx": cb, "kept": k}, dtype=np.int32)
                        .groupby("cell_idx", sort=False)["kept"].sum()
                        .reset_index()
                    )
                    reads_kept_per_cell_list[j][pc_reads["cell_idx"].to_numpy()] += pc_reads["kept"].to_numpy(np.int64)
                    kept_reads_total_list[j] += int(pc_reads["kept"].sum())

                # UMI survival at this depth (>=1 kept read)
                survive = (k > 0)
                if survive.any():
                    sel_cells = cb[survive]
                    sel_genes = gb[survive]

                    # each surviving family contributes 1 UMI to (cell, gene)
                    df_chunk = pd.DataFrame(
                        {"cell_idx": sel_cells, "gene_idx": sel_genes, "umis": 1},
                        dtype=np.int32
                    )
                    agg = df_chunk.groupby(["cell_idx","gene_idx"], sort=False)["umis"].sum().reset_index()

                    grouped_per_depth[j].append((
                        agg["cell_idx"].to_numpy(np.int32),
                        agg["gene_idx"].to_numpy(np.int32),
                        agg["umis"].to_numpy(np.int32),
                    ))

                    pc_umis = (
                        pd.DataFrame({"cell_idx": sel_cells, "one": 1}, dtype=np.int32)
                        .groupby("cell_idx", sort=False)["one"].sum()
                        .reset_index()
                    )
                    umis_surv_per_cell_list[j][pc_umis["cell_idx"].to_numpy()] += pc_umis["one"].to_numpy(np.int64)
                    kept_umis_total_list[j] += int(pc_umis["one"].sum())

                p_prev = p  # advance ladder

        dt = time.time() - s0
        last_reads = kept_reads_total_list[K-1] if K > 0 else 0
        print(f"  [shard {si:02d}/{len(shards)}] processed in {dt:.1f}s | cum_reads@max≈{last_reads:,}")

    # ----- After all shards for this replicate: build per-depth AnnData & write -----
    for t, p_target in zip(target_rpcs, p_for_target):
        j = _p_to_index(p_target)
        tag = f"rpc{t:d}"
        out_path = os.path.join(SAVE_DIR, f"subsample_MAPPED_{tag}_r{rep}.h5ad")
        if os.path.exists(out_path):
            print(f"[skip] exists → {out_path}")
            index_rows.append(dict(target_rpc=int(t), rep=int(rep), h5ad=out_path, p_keep=float(p_target)))
            continue

        # Build counts matrix
        X = _build_counts_matrix(n_cells, n_genes, grouped_per_depth[j])

        # Compose AnnData (reuse reference metadata to keep identity)
        ad = sc.AnnData(
            X=X,
            obs=ad_ref.obs.copy(),
            var=ad_ref.var.copy(),
            obsm=ad_ref.obsm.copy() if ad_ref.obsm is not None else None,
            uns=ad_ref.uns.copy() if ad_ref.uns is not None else None,
        )

        # Per-cell stats + saturation (10x definition: 1 - UMIs/reads)
        reads_kept = reads_kept_per_cell_list[j]
        umis_surv  = umis_surv_per_cell_list[j]
        ad.obs[f"reads_sim_MAPPED_{tag}"] = reads_kept
        ad.obs[f"umis_sim_MAPPED_{tag}"]  = umis_surv
        with np.errstate(divide="ignore", invalid="ignore"):
            sat = 1.0 - (umis_surv.astype(float) / np.clip(reads_kept, 1, None))
            sat[reads_kept == 0] = np.nan
        ad.obs[f"saturation_sim_MAPPED_{tag}"] = sat

        # Provenance
        ad.uns.setdefault("subsample_meta", {})
        ad.uns["subsample_meta"].update({
            "target_rpc": int(t),
            "rep": int(rep),
            "p_keep": float(p_target),
            "reads_in_shards": int(reads_in_shards),
            "n_cells": int(n_cells),
            "n_genes": int(n_genes),
            "seed_base": int(RNG_SEED_BASE),
            "nested_depths": [float(x) for x in p_grid.tolist()],
        })

        _ensure_lognorm(ad, LAYER_LOGNORM)
        ad.write_h5ad(out_path)
        print(f"[done] wrote {out_path} | kept_reads={kept_reads_total_list[j]:,} | kept_UMIs={kept_umis_total_list[j]:,}")

        index_rows.append(dict(target_rpc=int(t), rep=int(rep), h5ad=out_path, p_keep=float(p_target)))

# Save an index for discovery later (includes 'rep'); guard empty
index_csv = os.path.join(SAVE_DIR, "subsample_runs_index.csv")
if len(index_rows) == 0:
    print("[warn] No subsamples were created (index is empty).")
    # write an empty index with expected columns to avoid downstream KeyError
    pd.DataFrame(columns=["target_rpc","rep","h5ad","p_keep"]).to_csv(index_csv, index=False)
else:
    pd.DataFrame(index_rows).sort_values(["target_rpc","rep"]).to_csv(index_csv, index=False)
    print(f"\n[save] run index → {index_csv}")


In [ ]:
# ============================================
# Cell 4: QC metrics + plots (replicate-aware) and artifact schema
# ============================================
RUNS_DIR    = os.path.join(OUT_DIR, "runs_rpc_clamped")
METRICS_DIR = os.path.join(OUT_DIR, "metrics_rpc_clamped")
os.makedirs(METRICS_DIR, exist_ok=True)

index_csv = os.path.join(RUNS_DIR, "subsample_runs_index.csv")
assert os.path.exists(index_csv), f"Run index not found: {index_csv}"
# -------- Load runs index, normalize dtypes --------
runs = pd.read_csv(index_csv)
if runs.empty:
    raise RuntimeError("Run index is empty — nothing to plot. Please run Cell 3 first.")

required_cols = {"h5ad", "p_keep", "rep", "target_rpc"}
missing = required_cols - set(runs.columns)
if missing:
    raise RuntimeError(f"Run index missing columns: {missing}")

runs["target_rpc"] = runs["target_rpc"].astype(int)
runs["rep"] = runs["rep"].astype(int)
runs = runs.sort_values(["target_rpc", "rep"]).reset_index(drop=True)

print("[index]")
print(runs)

# -------- Helpers --------
def _per_cell_genes(ad):
    X = ad.X
    if sparse.issparse(X):
        return np.asarray(X.getnnz(axis=1)).ravel()
    return (np.asarray(X) > 0).sum(axis=1)

def _metrics_for_run(h5ad_path, target_rpc, rep):
    ad = sc.read_h5ad(h5ad_path)
    tag = f"rpc{int(target_rpc)}"
    rcol = f"reads_sim_MAPPED_{tag}"
    ucol = f"umis_sim_MAPPED_{tag}"
    scol = f"saturation_sim_MAPPED_{tag}"

    for c in (rcol, ucol, scol):
        if c not in ad.obs.columns:
            raise RuntimeError(f"Missing expected column in {os.path.basename(h5ad_path)}: {c}")

    reads_per_cell = ad.obs[rcol].to_numpy(dtype=float)
    umis_per_cell  = ad.obs[ucol].to_numpy(dtype=float)
    genes_per_cell = _per_cell_genes(ad).astype(float)
    sat_per_cell   = ad.obs[scol].to_numpy(dtype=float)

    total_reads = float(np.nansum(reads_per_cell))
    total_umis  = float(np.nansum(umis_per_cell))

    out = dict(
        target_rpc=int(target_rpc),
        rep=int(rep),
        h5ad=h5ad_path,
        p_keep=float(ad.uns.get("subsample_meta", {}).get("p_keep", np.nan)),
        n_cells=int(ad.n_obs),
        total_reads=total_reads,
        total_umis=total_umis,
        median_reads=float(np.nanmedian(reads_per_cell)),
        mean_reads=float(np.nanmean(reads_per_cell)),
        median_umis=float(np.nanmedian(umis_per_cell)),
        mean_umis=float(np.nanmean(umis_per_cell)),
        median_genes=float(np.nanmedian(genes_per_cell)),
        mean_genes=float(np.nanmean(genes_per_cell)),
        median_saturation=float(np.nanmedian(sat_per_cell)),
        mean_saturation=float(np.nanmean(sat_per_cell)),
    )

    # lightweight consistency check (not fatal)
    try:
        umis_from_X = float(np.asarray(ad.X.sum(axis=1)).ravel().sum())
        if abs(umis_from_X - total_umis) > 0.01 * max(1.0, total_umis):
            print(f"[warn] UMIs mismatch (X vs obs) for {os.path.basename(h5ad_path)}: "
                  f"{umis_from_X:.0f} vs {total_umis:.0f}")
    except Exception as e:
        print(f"[note] skipped UMIs-from-X check: {e}")

    return out

def _agg_mean_sd(df, cols, xcol="reads_millions"):
    g = df.groupby("target_rpc", as_index=False)
    base = g[xcol].agg(x_mean="mean", x_sd="std")
    out = base.copy()
    for c in cols:
        agg = g[c].agg(**{f"{c}_mean": "mean", f"{c}_sd": "std"})
        out = out.merge(agg, on="target_rpc", how="left")
    out = out.merge(g.size().rename(columns={"size": "n"}), on="target_rpc", how="left")
    return out.sort_values("x_mean")

# -------- Compute per-rep metrics over all runs --------
rows = []
for _, r in runs.iterrows():
    print(f"[metrics] rpc={int(r['target_rpc'])} rep={int(r['rep'])}  {os.path.basename(r['h5ad'])}")
    rows.append(_metrics_for_run(r["h5ad"], r["target_rpc"], r["rep"]))

perrep_df = pd.DataFrame(rows).sort_values(["target_rpc","rep"]).reset_index(drop=True)
perrep_df["reads_millions"] = perrep_df["total_reads"] / 1e6

# -------- Aggregate across replicates (mean ± SD) --------
metric_cols = [
    "median_reads","median_umis","median_genes","median_saturation",
    "mean_reads","mean_umis","mean_genes","mean_saturation",
    "total_reads","n_cells"
]
agg_df = _agg_mean_sd(perrep_df, metric_cols, xcol="reads_millions")
agg_df["reads_millions_mean"] = agg_df["x_mean"]
agg_df["reads_millions_sd"]   = agg_df["x_sd"]
agg_df = agg_df.drop(columns=["x_mean","x_sd"])
# also expose a plain "reads_millions" column on the aggregate (mean) for downstream consumers
agg_df["reads_millions"] = agg_df["reads_millions_mean"]

# -------- Save CSVs (per-rep + aggregated) --------
perrep_csv = os.path.join(METRICS_DIR, "metrics_rpc_clamped_perrep.csv")
agg_csv    = os.path.join(METRICS_DIR, "metrics_rpc_clamped_agg.csv")
compat_csv = os.path.join(METRICS_DIR, "metrics_rpc_clamped.csv")  # alias for downstream code

perrep_df.to_csv(perrep_csv, index=False)
agg_df.to_csv(agg_csv, index=False)
agg_df.to_csv(compat_csv, index=False)

print(f"[save] per-rep metrics   → {perrep_csv}")
print(f"[save] aggregated metrics→ {agg_csv}")
print(f"[save] (compat alias)   → {compat_csv}")

# -------- Plotting (medians; per-rep scatter + mean±SD error bars) --------
plt.figure(figsize=(12, 9))
plots = [
    ("median_reads_mean",      "median_reads_sd",      "Median reads/cell"),
    ("median_umis_mean",       "median_umis_sd",       "Median UMIs/cell"),
    ("median_genes_mean",      "median_genes_sd",      "Median genes/cell"),
    ("median_saturation_mean", "median_saturation_sd", "Median saturation"),
]
for i, (col_mean, col_sd, label) in enumerate(plots, 1):
    ax = plt.subplot(2, 2, i)
    base_col = col_mean.replace("_mean", "")
    ax.scatter(perrep_df["reads_millions"], perrep_df[base_col], s=15, alpha=0.4)
    ax.errorbar(
        agg_df["reads_millions_mean"], agg_df[col_mean], yerr=agg_df[col_sd],
        fmt="-o", capsize=3
    )
    ax.set_xlabel("Total reads kept (millions)")
    ax.set_ylabel(label)
    ax.grid(True, alpha=0.3)
plt.suptitle("Subsampling (RPC-clamped): medians vs total kept reads (mean±SD across reps)", y=1.02)
plt.tight_layout()
panel_med_png = os.path.join(METRICS_DIR, "panel_medians_vs_total_reads.png")
plt.savefig(panel_med_png, dpi=180, bbox_inches="tight")
plt.close()
print(f"[save] {panel_med_png}")

# -------- Plotting (means; per-rep scatter + mean±SD error bars) --------
plt.figure(figsize=(12, 9))
plots_mean = [
    ("mean_reads_mean",      "mean_reads_sd",      "Mean reads/cell"),
    ("mean_umis_mean",       "mean_umis_sd",       "Mean UMIs/cell"),
    ("mean_genes_mean",      "mean_genes_sd",      "Mean genes/cell"),
    ("mean_saturation_mean", "mean_saturation_sd", "Mean saturation"),
]
for i, (col_mean, col_sd, label) in enumerate(plots_mean, 1):
    ax = plt.subplot(2, 2, i)
    base_col = col_mean.replace("_mean", "")
    ax.scatter(perrep_df["reads_millions"], perrep_df[base_col], s=15, alpha=0.4)
    ax.errorbar(
        agg_df["reads_millions_mean"], agg_df[col_mean], yerr=agg_df[col_sd],
        fmt="-o", capsize=3
    )
    ax.set_xlabel("Total reads kept (millions)")
    ax.set_ylabel(label)
    ax.grid(True, alpha=0.3)
plt.suptitle("Subsampling (RPC-clamped): means vs total kept reads (mean±SD across reps)", y=1.02)
plt.tight_layout()
panel_mean_png = os.path.join(METRICS_DIR, "panel_means_vs_total_reads.png")
plt.savefig(panel_mean_png, dpi=180, bbox_inches="tight")
plt.close()
print(f"[save] {panel_mean_png}")

# -------- Artifact schema (handoff to analysis_sub.ipynb) --------
# NOTE: structure matches the "runs_index"/"metrics" schema format expected by
# analysis_sub.ipynb's load_schema()/_coerce_schema() — both need a "path" key.
schema = dict(
    runs_index=dict(
        path=index_csv,
        columns=list(runs.columns),
        description="Index of subsample runs (target_rpc, p_keep, h5ad path, rep).",
        reads_col_pattern="reads_sim_MAPPED_rpc{target_rpc}",
        umis_col_pattern="umis_sim_MAPPED_rpc{target_rpc}",
        saturation_col_pattern="saturation_sim_MAPPED_rpc{target_rpc}",
        rep_column="rep",
        has_replicates=True,
    ),
    metrics=dict(
        path=compat_csv,                 # aggregated metrics (mean across reps); used by downstream plotting
        per_rep_path=perrep_csv,
        aggregated_path=agg_csv,
        per_rep_columns=list(perrep_df.columns),
        aggregated_columns=list(agg_df.columns),
        x_axis_aggregated="reads_millions_mean",
        primary_metrics=[
            "median_reads_mean","median_umis_mean","median_genes_mean","median_saturation_mean",
            "mean_reads_mean","mean_umis_mean","mean_genes_mean","mean_saturation_mean",
        ],
        description="Per-replicate QC metrics and aggregated (mean±SD) across replicates.",
    ),
    paths=dict(
        runs_dir=RUNS_DIR,
        metrics_dir=METRICS_DIR,
        figures=[panel_med_png, panel_mean_png],
    ),
    reference=dict(
        ref_anndata=REF_ADATA,
        note="Reference (deep run) AnnData used to fix HVGs/PCA/UMAP in subsequent steps.",
    ),
)
schema_json = os.path.join(METRICS_DIR, "artifact_schema.json")
with open(schema_json, "w") as f:
    json.dump(schema, f, indent=2)
print(f"[save] schema → {schema_json}")

print("\n[done] Replicate-aware QC metrics computed and plots saved.")
print(f"[handoff] point analysis_sub.ipynb's SCHEMA_JSON at: {schema_json}")
